# Практична робота 04: Проєктування рішення навколо фіксованого класифікатора

## Набір даних Hill-Valley (UCI)

Мета: зафіксувати модель, отримати OOF-бали, підготувати основу для аналізу порогів.

**Увага:** на цьому етапі ми не виконуємо повний цикл (сітка порогів, калібрування, політика, test).
Лише фіксація протоколу та отримання OOF-балів.


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

### 1. Завантаження даних

Використовуємо `ucimlrepo` для отримання набору Hill-Valley (id=166).
Обираємо версію **без шуму** (Hill_Valley_without_noise) для чистоти експерименту.

In [15]:
import pandas as pd

# 1. Downloading file. adding header=None, 
hill_valley = pd.read_csv('Hill_Valley_without_noise_Training.data', header=None)

# 2. dividing data:
# iloc[:, :-1] takes all except last (cause it X)
X = hill_valley.iloc[:, :-1]

# iloc[:, -1] takes all except last (cause it y)
y = hill_valley.iloc[:, -1]

# 3. Outputing data
print('Форма X:', X.shape)
print('Форма y:', y.shape)
print('Розподіл класів:')
print(y.value_counts())
print('\nПерші 5 ознак:')
print(X.head())


Форма X: (607, 100)
Форма y: (607,)
Розподіл класів:
100
0        305
1        301
class      1
Name: count, dtype: int64

Перші 5 ознак:
            0            1            2            3            4   \
0           X1           X2           X3           X4           X5   
1  1317.265789  1315.220951  1312.770581  1309.834252  1306.315588   
2  7329.967624  7379.907443  7441.799231  7518.503422  7613.565031   
3  809.4214096  809.7801194  810.2071911  810.7156529   811.321016   
4  45334.20888  45334.21356  45334.21906   45334.2255  45334.23305   

            5            6            7            8            9   ...  \
0           X6           X7           X8           X9          X10  ...   
1  1302.099102  1297.046401  1290.991646  1283.736109  1275.041652  ...   
2  7731.377492  7877.385707  8058.337694  8282.596458  8560.526497  ...   
3  812.0417476  812.8998341  813.9214524   815.137768  816.5858856  ...   
4  45334.24191   45334.2523  45334.26448  45334.27876  45334.29552

### 2. Фіксація train/test split та random_state

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split


hill_valley = pd.read_csv('Hill_Valley_without_noise_Training.data') 


X = hill_valley.iloc[:, :-1]
y = hill_valley.iloc[:, -1]


RANDOM_STATE = 42
TEST_SIZE = 0.2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Розподіл класів у train:')
print(y_train.value_counts(normalize=True))
print('Розподіл класів у test:')
print(y_test.value_counts(normalize=True))


Train: (484, 100) Test: (122, 100)
Розподіл класів у train:
class
0    0.504132
1    0.495868
Name: proportion, dtype: float64
Розподіл класів у test:
class
1    0.5
0    0.5
Name: proportion, dtype: float64


### 3. Фіксована модель та Pipeline

Обираємо **LogisticRegression** як базовий класифікатор.
Preprocessing: StandardScaler у Pipeline.
Гіперпараметри: `C=1.0`, `solver='lbfgs'`, `max_iter=1000`.

Якщо у вашій лабораторній роботі №2 використовувалась інша модель (наприклад, RandomForest),
замініть її тут, але зафіксуйте всі параметри.

In [18]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=RANDOM_STATE))
])

print('Pipeline зафіксовано:')
print(pipeline)

Pipeline зафіксовано:
Pipeline(steps=[('scaler', StandardScaler()),
                ('clf', LogisticRegression(max_iter=1000, random_state=42))])


### 4. Стратифікована крос-валідація та OOF-бали

Використовуємо `StratifiedKFold` з 5 фолдами.
Спосіб отримання балу: `predict_proba` (ймовірність класу 1).

Для кожного об'єкта тренувального набору отримуємо прогноз від моделі,
яка не навчалася на цьому об'єкті.

In [19]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Отримуємо OOF-ймовірності для всього train набору
oof_scores = cross_val_predict(
    pipeline, X_train, y_train,
    cv=cv,
    method='predict_proba',
    n_jobs=-1
)[:, 1]  # ймовірність класу 1 (hill)

# Перевіряємо, що кожен об'єкт має рівно один прогноз
assert len(oof_scores) == len(X_train), 'Кількість OOF-прогнозів не збігається з кількістю об\'єктів'

print('OOF-бали отримано. Приклад перших 10:')
print(oof_scores[:10])

OOF-бали отримано. Приклад перших 10:
[0.50952585 0.85534823 0.40540655 0.5083684  0.50832839 0.49829611
 0.52178756 0.50138154 0.49823641 0.50817557]


### 5. Формування таблиці OOF

Створюємо DataFrame з полями: `object_id`, `y_true`, `score`, `fold`.
`object_id` — індекс об'єкта у вихідному train наборі (0..N-1).
`fold` — номер фолду, у якому об'єкт був у валідаційній частині.

In [20]:
# Визначаємо fold для кожного об'єкта
fold_ids = np.zeros(len(X_train), dtype=int)
for fold_idx, (_, val_idx) in enumerate(cv.split(X_train, y_train)):
    fold_ids[val_idx] = fold_idx

oof_df = pd.DataFrame({
    'object_id': np.arange(len(X_train)),
    'y_true': y_train.values.ravel(),
    'score': oof_scores,
    'fold': fold_ids
})

print('Форма OOF-таблиці:', oof_df.shape)
print(oof_df.head(10))
print('\nРозподіл по фолдах:')
print(oof_df['fold'].value_counts().sort_index())

Форма OOF-таблиці: (484, 4)
   object_id  y_true     score  fold
0          0       1  0.509526     0
1          1       1  0.855348     0
2          2       0  0.405407     1
3          3       1  0.508368     1
4          4       0  0.508328     1
5          5       0  0.498296     2
6          6       0  0.521788     3
7          7       0  0.501382     4
8          8       0  0.498236     2
9          9       1  0.508176     2

Розподіл по фолдах:
fold
0    97
1    97
2    97
3    97
4    96
Name: count, dtype: int64


### 6. Базова оцінка OOF-балів

Обчислюємо ROC-AUC та accuracy за стандартним порогом 0.5.
Це проміжна оцінка, яка не є фінальною метрикою рішення.

In [22]:
roc_auc = roc_auc_score(oof_df['y_true'], oof_df['score'])
y_pred_default = (oof_df['score'] >= 0.5).astype(int)
acc = accuracy_score(oof_df['y_true'], y_pred_default)

print(f'OOF ROC-AUC: {roc_auc:.4f}')
print(f'OOF Accuracy (порог 0.5): {acc:.4f}')

OOF ROC-AUC: 0.7814
OOF Accuracy (порог 0.5): 0.6694


### 7. Збереження OOF-таблиці

Зберігаємо `oof_results.csv` у директорію `../data/` для подальшого аналізу.

In [24]:
import os
os.makedirs('data', exist_ok=True)
oof_df.to_csv('data/oof_results.csv', index=False)
print('OOF-таблицю збережено у ../data/oof_results.csv')

OOF-таблицю збережено у ../data/oof_results.csv


---

## Примітки

- `score` у таблиці — це ймовірність класу 1 (hill), отримана через `predict_proba`.
- `fold` вказує, у якому фолді об'єкт був у валідаційній вибірці.
- Наступні кроки (сітка порогів, аналіз FP/FN, калібрування) будуть виконані в окремих комірках або наступних ноутбуках.